# Lab 1.4: Exercise - Standard BPE vs Entropy-BPE

**Course:** Advanced Natural Language Processing (NLP702/806)

**Instructor:** Dr. Fajri Koto

---

## Task
Implement and compare **two BPE trainers/tokenizers** that differ only in **merge selection**.  
Train both on `train_texts.txt` to target vocab size **5000**, then evaluate on `test_texts.txt`.

## Pre-tokenization
Whitespace-split into words. Each word is a sequence of characters plus `</w>`.

## Merge selection

### 1) Standard BPE
Pick the adjacent pair `(X,Y)` with maximum frequency `count(XY)`.

### 2) Entropy-BPE (left-context entropy)
For each occurrence of adjacent `(X,Y)` in a word, define left context `L` as the symbol before `X`, else `<B>` at word start. Maintain counts `count(L,X,Y)`.

Compute:

$$
p(L \mid XY)=\frac{\text{count}(L,X,Y)}{\sum_{L'}\text{count}(L',X,Y)}
$$

$$
H(XY)=-\sum_L p(L\mid XY)\log_2 p(L\mid XY)
$$

Select merge by **min** $H(XY)$, tie-break by **max** `count(XY)`.

Stop when vocab size reaches **5000** (initial alphabet + merges).

## Evaluation (test set)

1.	**Compression:** tokens per character on the test set.
2.	**UNK rate** (no bytes): percent of tokens that are UNK, where UNK means any character not seen in the training alphabet.
3.	**Morph boundary heuristic:** choose your own suffix list (and state it). For test words that match stem+suffix, report the fraction where tokenization keeps a boundary at the stem–suffix split.

## Theory 
Answer:
1) Why entropy-based merges can help morphologically rich languages.
2) Big-O merge-selection cost for Standard vs Entropy-BPE + what extra stats Entropy-BPE maintains.

## Submit
- Code (both trainers + eval)
- ≤200-word report with metrics + 1–2 sentence interpretation + complexity

In [4]:
train_path = "data/corpus/train_texts.txt"
test_path = "data/corpus/test_texts.txt"

with open(train_path, "r") as f:
    train_texts = f.readlines()

with open(test_path, "r") as f:
    test_texts = f.readlines()

In [5]:
# Shared implementation: Standard BPE vs Entropy-BPE
# The two trainers differ ONLY in merge selection.

from collections import Counter
import math
import time
import pandas as pd

EOW = "</w>"
WORD_START = "<B>"
TARGET_VOCAB_SIZE = 5000

# You can edit this suffix list (state it in your report).
SUFFIXES = [
    "ingly", "edly", "ness", "ment", "tion", "sion", "able", "ible", "less", "ful",
    "est", "ers", "ies", "ing", "ed", "ly", "er", "es", "s"
]


def whitespace_tokenize(texts):
    return [line.strip().split() for line in texts]


def word_to_symbols(word):
    return tuple(list(word) + [EOW])


def build_word_sequences(tokenized_lines):
    # Counter of word symbol sequences with word frequency.
    sequences = Counter()
    for line in tokenized_lines:
        for word in line:
            sequences[word_to_symbols(word)] += 1
    return sequences


def merge_vocab(pair, word_sequences):
    # Merge all occurrences of a selected adjacent pair.
    a, b = pair
    merged_symbol = a + b
    out = Counter()

    for seq, freq in word_sequences.items():
        new_seq = []
        i = 0
        n = len(seq)
        while i < n:
            if i + 1 < n and seq[i] == a and seq[i + 1] == b:
                new_seq.append(merged_symbol)
                i += 2
            else:
                new_seq.append(seq[i])
                i += 1
        out[tuple(new_seq)] += freq

    return out


def get_pair_freq(word_sequences):
    pair_freq = Counter()
    for seq, freq in word_sequences.items():
        for i in range(len(seq) - 1):
            pair_freq[(seq[i], seq[i + 1])] += freq
    return pair_freq


def get_entropy_stats(word_sequences):
    # count(XY) and count(L, X, Y)
    pair_freq = Counter()
    left_context_counts = {}

    for seq, freq in word_sequences.items():
        prev = WORD_START
        n = len(seq)
        for i in range(n - 1):
            x = seq[i]
            y = seq[i + 1]
            pair = (x, y)

            pair_freq[pair] += freq

            lmap = left_context_counts.get(pair)
            if lmap is None:
                left_context_counts[pair] = {prev: freq}
            else:
                lmap[prev] = lmap.get(prev, 0) + freq

            prev = x

    return pair_freq, left_context_counts


def choose_pair_standard(pair_freq):
    # max count(XY), deterministic lexicographic tie-break
    return max(pair_freq, key=lambda p: (pair_freq[p], p))


def choose_pair_entropy(pair_freq, left_context_counts):
    # min H(XY), tie-break by max count(XY), then lexicographic
    best_pair = None
    best_h = float("inf")
    best_freq = -1

    for pair, lcounts in left_context_counts.items():
        total = pair_freq[pair]
        inv_total = 1.0 / total
        h = 0.0
        for c in lcounts.values():
            p = c * inv_total
            h -= p * math.log2(p)

        if h < best_h:
            best_h = h
            best_pair = pair
            best_freq = total
        elif h == best_h:
            if total > best_freq:
                best_pair = pair
                best_freq = total
            elif total == best_freq and pair < best_pair:
                best_pair = pair

    return best_pair


def train_bpe(
    tokenized_lines,
    target_vocab_size=TARGET_VOCAB_SIZE,
    mode="standard",
    verbose=False,
    print_every=50,
    run_label=None,
):
    if mode not in {"standard", "entropy"}:
        raise ValueError("mode must be 'standard' or 'entropy'")

    word_sequences = build_word_sequences(tokenized_lines)

    vocab = set()
    for seq in word_sequences:
        vocab.update(seq)
    initial_vocab_size = len(vocab)

    merges = []
    start_time = time.time()
    label = run_label if run_label is not None else mode

    if verbose:
        print(
            f"[{label}] start: initial_vocab={initial_vocab_size}, target_vocab={target_vocab_size}",
            flush=True,
        )

    # Stop condition required by the task:
    # initial alphabet + merges reaches target vocab size.
    while initial_vocab_size + len(merges) < target_vocab_size:
        if mode == "standard":
            pair_freq = get_pair_freq(word_sequences)
            if not pair_freq:
                break
            best_pair = choose_pair_standard(pair_freq)
        else:
            pair_freq, left_context_counts = get_entropy_stats(word_sequences)
            if not pair_freq:
                break
            best_pair = choose_pair_entropy(pair_freq, left_context_counts)

        merges.append(best_pair)
        word_sequences = merge_vocab(best_pair, word_sequences)

        if verbose:
            merge_count = len(merges)
            if merge_count == 1 or merge_count % print_every == 0:
                current_vocab = initial_vocab_size + merge_count
                elapsed = time.time() - start_time
                rate = merge_count / elapsed if elapsed > 0 else 0.0
                remaining = max(target_vocab_size - current_vocab, 0)
                eta = remaining / rate if rate > 0 else float("inf")
                eta_str = f"{eta:.1f}s" if eta != float("inf") else "inf"
                print(
                    f"[{label}] merges={merge_count} | vocab={current_vocab}/{target_vocab_size} | "
                    f"elapsed={elapsed:.1f}s | eta={eta_str}",
                    flush=True,
                )

    if verbose:
        total_time = time.time() - start_time
        final_vocab = initial_vocab_size + len(merges)
        print(
            f"[{label}] done: merges={len(merges)}, final_vocab={final_vocab}, time={total_time:.1f}s",
            flush=True,
        )

    return merges, initial_vocab_size


def apply_merges_to_word(word, merges):
    symbols = list(word) + [EOW]
    for pair in merges:
        a, b = pair
        merged = a + b
        new_symbols = []
        i = 0
        n = len(symbols)
        while i < n:
            if i + 1 < n and symbols[i] == a and symbols[i + 1] == b:
                new_symbols.append(merged)
                i += 2
            else:
                new_symbols.append(symbols[i])
                i += 1
        symbols = new_symbols
    return symbols


def build_train_alphabet(train_tok):
    alphabet = set()
    for sent in train_tok:
        for word in sent:
            alphabet.update(word)
    return alphabet


def token_has_unseen_char(token, train_alphabet):
    raw = token.replace(EOW, "")
    return any(ch not in train_alphabet for ch in raw)


def morph_boundary_score(test_words, tokenized_words, suffixes):
    # Fraction of stem+suffix words that keep a token boundary at stem|suffix.
    matches = 0
    keeps = 0
    suffixes_sorted = sorted(suffixes, key=len, reverse=True)

    for word, symbols in zip(test_words, tokenized_words):
        for suf in suffixes_sorted:
            if word.endswith(suf) and len(word) > len(suf):
                stem = word[:-len(suf)]
                matches += 1

                prefix = ""
                boundary_kept = False
                for sym in symbols[:-1]:
                    prefix += sym.replace(EOW, "")
                    if prefix == stem:
                        boundary_kept = True
                        break
                    if len(prefix) > len(stem):
                        break

                if boundary_kept:
                    keeps += 1
                break

    frac = keeps / matches if matches else None
    return keeps, matches, frac


def evaluate_tokenizer(name, merges, train_tok, test_tok, suffixes):
    test_words = [w for sent in test_tok for w in sent]
    tokenized_words = [apply_merges_to_word(w, merges) for w in test_words]

    # 1) Compression: tokens per character
    total_tokens = sum(len(symbols) for symbols in tokenized_words)
    total_chars = sum(len(w) for w in test_words)
    tokens_per_char = total_tokens / total_chars if total_chars else 0.0

    # 2) UNK rate: percent of BPE tokens containing unseen training characters
    train_alphabet = build_train_alphabet(train_tok)
    unk_tokens = sum(
        1
        for symbols in tokenized_words
        for token in symbols
        if token_has_unseen_char(token, train_alphabet)
    )
    unk_rate_pct = 100.0 * unk_tokens / total_tokens if total_tokens else 0.0

    # 3) Morph boundary heuristic
    keeps, matches, frac = morph_boundary_score(test_words, tokenized_words, suffixes)

    return {
        "Tokenizer": name,
        "Merges": len(merges),
        "Tokens/Char (lower better)": round(tokens_per_char, 6),
        "UNK Rate % (lower better)": round(unk_rate_pct, 4),
        "Morph Boundary % (higher better)": round(100.0 * frac, 2) if frac is not None else None,
        "Morph Matches": f"{keeps}/{matches}",
    }




In [6]:
# Train both tokenizers on train_texts.txt (target vocab size = 5000)

train_tok = whitespace_tokenize(train_texts)
test_tok = whitespace_tokenize(test_texts)

standard_merges, standard_initial_vocab_size = train_bpe(
    train_tok,
    target_vocab_size=TARGET_VOCAB_SIZE,
    mode="standard",
    verbose=True,
    print_every=100,
    run_label="Standard BPE",
)
entropy_bpe_merges, entropy_initial_vocab_size = train_bpe(
    train_tok,
    target_vocab_size=TARGET_VOCAB_SIZE,
    mode="entropy",
    verbose=True,
    print_every=25,
    run_label="Entropy-BPE",
)

print("Training summary")
print("-" * 80)
print(
    f"Standard BPE: initial_vocab={standard_initial_vocab_size}, "
    f"merges={len(standard_merges)}, "
    f"final_vocab={standard_initial_vocab_size + len(standard_merges)}"
)
print(
    f"Entropy-BPE: initial_vocab={entropy_initial_vocab_size}, "
    f"merges={len(entropy_bpe_merges)}, "
    f"final_vocab={entropy_initial_vocab_size + len(entropy_bpe_merges)}"
)
print(f"Standard first 10 merges: {standard_merges[:10]}")
print(f"Entropy first 10 merges: {entropy_bpe_merges[:10]}")




[Standard BPE] start: initial_vocab=3917, target_vocab=5000
[Standard BPE] merges=1 | vocab=3918/5000 | elapsed=1.5s | eta=1591.8s
[Standard BPE] merges=100 | vocab=4017/5000 | elapsed=125.0s | eta=1228.3s
[Standard BPE] merges=200 | vocab=4117/5000 | elapsed=237.3s | eta=1047.7s
[Standard BPE] merges=300 | vocab=4217/5000 | elapsed=343.4s | eta=896.3s
[Standard BPE] merges=400 | vocab=4317/5000 | elapsed=445.7s | eta=761.1s
[Standard BPE] merges=500 | vocab=4417/5000 | elapsed=545.4s | eta=635.9s
[Standard BPE] merges=600 | vocab=4517/5000 | elapsed=643.0s | eta=517.6s
[Standard BPE] merges=700 | vocab=4617/5000 | elapsed=738.4s | eta=404.0s
[Standard BPE] merges=800 | vocab=4717/5000 | elapsed=832.8s | eta=294.6s
[Standard BPE] merges=900 | vocab=4817/5000 | elapsed=925.8s | eta=188.2s
[Standard BPE] merges=1000 | vocab=4917/5000 | elapsed=1018.0s | eta=84.5s
[Standard BPE] done: merges=1083, final_vocab=5000, time=1093.7s
[Entropy-BPE] start: initial_vocab=3917, target_vocab=5000
[E

In [7]:
# Evaluate both on test_texts.txt

results = [
    evaluate_tokenizer("Standard BPE", standard_merges, train_tok, test_tok, SUFFIXES),
    evaluate_tokenizer("Entropy-BPE", entropy_bpe_merges, train_tok, test_tok, SUFFIXES),
]

results_df = pd.DataFrame(results)
print(f"Suffix list used: {SUFFIXES}")
print(results_df.to_string(index=False))



Suffix list used: ['ingly', 'edly', 'ness', 'ment', 'tion', 'sion', 'able', 'ible', 'less', 'ful', 'est', 'ers', 'ies', 'ing', 'ed', 'ly', 'er', 'es', 's']
   Tokenizer  Merges  Tokens/Char (lower better)  UNK Rate % (lower better)  Morph Boundary % (higher better) Morph Matches
Standard BPE    1083                    0.534916                     0.0215                             48.15   30709/63778
 Entropy-BPE    1083                    1.122402                     0.0102                             99.91   63723/63778
